## Mean entre histgb, extra trees, random forest

### Description of the Technique

Mean ensembling is a simple ensemble learning technique in which the final prediction is obtained by taking the **arithmetic average of the predictions** produced by multiple base models. Formally, given ( M ) trained models producing predictions.

In this work, the ensemble combines **HistGradientBoosting**, **Random Forest**, and **Extra Trees** regressors, each trained independently on the same feature space.

---

### Rationale Behind Mean Ensembling

The core motivation of mean ensembling is **variance reduction**. While individual models may produce noisy or biased predictions, averaging multiple models smooths out individual errors, provided that their errors are not perfectly correlated.

Tree-based ensemble models naturally satisfy this condition because:

* They rely on different sources of randomness,
* They learn different internal structures,
* They exhibit distinct bias–variance trade-offs.

As a result, averaging their predictions leads to a more stable and robust estimator.

---

### Why These Models Work Well Together

Although all base learners belong to the family of tree-based models, they differ substantially in how they learn:

* **HistGradientBoosting** builds trees sequentially, focusing on correcting previous errors, resulting in low bias but potentially higher variance.
* **Random Forests** rely on bagging and feature subsampling, strongly reducing variance.
* **Extra Trees** introduce additional randomness in split selection, further decorrelating trees and improving generalization.

Because these models make **different types of errors**, their combination through averaging is effective even without complex weighting or stacking mechanisms.

---

### Advantages Over More Complex Ensembling Methods

Mean ensembling offers several practical advantages:

* **No additional training stage** is required.
* **No risk of meta-model overfitting**, unlike stacking approaches.
* **Computationally efficient**, especially compared to neural or multi-level ensembles.
* **Easy to interpret and reproduce**, with minimal hyperparameter sensitivity.

For structured tabular regression problems, mean ensembling often captures most of the achievable ensemble gain with a fraction of the complexity.

---

### Suitability for the Price Prediction Task

In price prediction problems, model errors tend to be **heterogeneous across the feature space**. Some models may perform better for newer vehicles, others for high-mileage or rare configurations. Mean ensembling naturally balances these local strengths, producing predictions that are less extreme and more reliable overall.

This makes the approach particularly well-suited for real-world regression tasks where stability and generalization are more important than marginal gains from highly complex ensemble strategies.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error
import os
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

In [2]:
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def bias(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) - np.asarray(y_pred)))

def get_random_configs(n_iter: int):
    configs = []
    keys = list(param_distributions.keys())
    for i in range(n_iter):
        params = {k: random.choice(list(param_distributions[k])) for k in keys}

        name = (
            f"R{i}_iter{params['max_iter']}"
            f"_lr{float(params['learning_rate']):.3f}"
            f"_l2{float(params['l2_regularization']):.2f}"
            f"_leaf{params['max_leaf_nodes']}"
        )
        configs.append((name, params))
    return configs

In [3]:
# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# at this point, this are all the features in use 
categorical_features = ['Brand', 'model', 'transmission', 'fuelType']              
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

N_SPLITS = 8 # the number of folds for K-Fold cross-validation
RANDOM_STATE = 42 # seed to control randomness

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


modelo normal, features todas, sem fe/fs price normal

In [ ]:
# =========================================================
# K-FOLD CROSS-VALIDATION
# =========================================================

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# =========================================================
# FIXED MODEL CONFIGURATION
# =========================================================

params = {
    "hgb": {
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

# =========================================================
# LOGGING
# =========================================================

log_path = "mean_complete_results.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =========================================")
    log("# SINGLE CONFIGURATION CROSS-VALIDATION")
    log("# ENSEMBLE: HGB + RF + ET (WEIGHTED MEAN)")
    log("# =========================================")
    log(f"Parameters: {params}")
    log("")

    # -----------------------------------------------------
    # METRIC CONTAINERS
    # -----------------------------------------------------

    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []

    # =====================================================
    # K-FOLD LOOP
    # =====================================================

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

        log(f"\n===== FOLD {fold}/{N_SPLITS} =====")

        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

        # -------------------------------------------------
        # PREPROCESSING
        # -------------------------------------------------

        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val, year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val, mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val, engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val, mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val, owners_state)

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val, model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val, transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

        # -------------------------------------------------
        # ENCODING
        # -------------------------------------------------

        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_final = pd.concat([X_train[numeric_features], X_train_high, X_train_low], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features],   X_val_high,   X_val_low], axis=1)

        # -------------------------------------------------
        # SCALING
        # -------------------------------------------------

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_final)
        X_val_scaled   = scaler.transform(X_val_final)

        # -------------------------------------------------
        # MODELS
        # -------------------------------------------------

        hgb = HistGradientBoostingRegressor(**params["hgb"])
        rf  = RandomForestRegressor(**params["rf"])
        et  = ExtraTreesRegressor(**params["et"])

        hgb.fit(X_train_scaled, y_train)
        rf.fit(X_train_scaled, y_train)
        et.fit(X_train_scaled, y_train)

        # -------------------------------------------------
        # ENSEMBLE PREDICTION
        # -------------------------------------------------

        y_tr = 0.6*hgb.predict(X_train_scaled) + 0.4*rf.predict(X_train_scaled)
        y_val_pred = 0.6*hgb.predict(X_val_scaled) + 0.4*rf.predict(X_val_scaled)

        # -------------------------------------------------
        # METRICS
        # -------------------------------------------------

        mae_tr  = mean_absolute_error(y_train, y_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, y_tr))
        r2_tr   = r2_score(y_train, y_tr)
        bias_tr = np.mean(y_train - y_tr)

        mae_val  = mean_absolute_error(y_val, y_val_pred)
        rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
        r2_val   = r2_score(y_val, y_val_pred)
        bias_val = np.mean(y_val - y_val_pred)
        med_ae   = median_absolute_error(y_val, y_val_pred)

        log(f"[TRAIN] R2={r2_tr:.4f} | RMSE={rmse_tr:.0f} | MAE={mae_tr:.0f} | Bias={bias_tr:.1f}")
        log(f"[VAL]   R2={r2_val:.4f} | RMSE={rmse_val:.0f} | MAE={mae_val:.0f} | Bias={bias_val:.1f}")

        fold_maes_tr.append(mae_tr)
        fold_rmses_tr.append(rmse_tr)
        fold_r2s_tr.append(r2_tr)
        fold_bias_tr.append(bias_tr)

        fold_maes_val.append(mae_val)
        fold_rmses_val.append(rmse_val)
        fold_r2s_val.append(r2_val)
        fold_bias_val.append(bias_val)
        fold_med_ae.append(med_ae)

    # =====================================================
    # FINAL RESULTS
    # =====================================================

    log("\n===== FINAL CROSS-VALIDATION RESULTS =====")
    log(f"[TRAIN AVG] MAE={np.mean(fold_maes_tr):.1f} | R2={np.mean(fold_r2s_tr):.4f} | Bias={np.mean(fold_bias_tr):.1f}")
    log(f"[VAL AVG]   MAE={np.mean(fold_maes_val):.1f} | "
        f"R2={np.mean(fold_r2s_val):.4f} | "
        f"Bias={np.mean(fold_bias_val):.1f} | "
        f"RMSE={np.mean(fold_rmses_val):.1f}")


o modelo base mas sem previous owners, log do price

In [ ]:
# =========================================================
# K-FOLD CROSS-VALIDATION
# =========================================================

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

DROP_FROM_MODEL = ["previousOwners"]

#este numeric features ta certo? nao falta year?
numeric_features = ["mileage", "engineSize", "tax", "mpg"]

# =========================================================
# FIXED MODEL CONFIGURATION
# =========================================================

params = {
    "hgb": {
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

# =========================================================
# LOGGING
# =========================================================

log_path = "single_config_no_previous_cv_results.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# ===============================================")
    log("# SINGLE CONFIG CV — WITHOUT previousOwners")
    log("# ENSEMBLE: HGB + RF + ET (WEIGHTED MEAN)")
    log("# ===============================================")
    log(f"Parameters: {params}")
    log("")

    # -----------------------------------------------------
    # METRIC CONTAINERS
    # -----------------------------------------------------

    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []

    # =====================================================
    # K-FOLD LOOP
    # =====================================================

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

        log(f"\n===== FOLD {fold}/{N_SPLITS} =====")

        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()
        y_train_log = np.log1p(y_train)

        # -------------------------------------------------
        # PREPROCESSING
        # -------------------------------------------------

        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val, year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val, mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val, engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val, "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val, mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val, owners_state)

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val, model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val, transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

        # DROP previousOwners
        X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
        X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

        # -------------------------------------------------
        # ENCODING
        # -------------------------------------------------

        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_final = pd.concat(
            [X_train[numeric_features], X_train_high, X_train_low], axis=1
        )
        X_val_final = pd.concat(
            [X_val[numeric_features], X_val_high, X_val_low], axis=1
        )

        # -------------------------------------------------
        # SCALING
        # -------------------------------------------------

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_final)
        X_val_scaled   = scaler.transform(X_val_final)

        if fold == 1:
            log(f"  > Features Used ({X_train_final.shape[1]})")

        # -------------------------------------------------
        # MODELS
        # -------------------------------------------------

        hgb = HistGradientBoostingRegressor(**params["hgb"])
        rf  = RandomForestRegressor(**params["rf"])
        et  = ExtraTreesRegressor(**params["et"])

        hgb.fit(X_train_final, y_train_log)
        rf.fit(X_train_final, y_train_log)
        et.fit(X_train_final, y_train_log)

        pred_tr = (
            0.6 * np.expm1(hgb.predict(X_train_final)) +
            0.4 * np.expm1(rf.predict(X_train_final))
        )

        pred_val = (
            0.6 * np.expm1(hgb.predict(X_val_final)) +
            0.4 * np.expm1(rf.predict(X_val_final))
        )
        # -------------------------------------------------
        # METRICS
        # -------------------------------------------------

        mae_tr  = mean_absolute_error(y_train, y_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, y_tr))
        r2_tr   = r2_score(y_train, y_tr)
        bias_tr = np.mean(y_train - y_tr)

        mae_val  = mean_absolute_error(y_val, y_val_pred)
        rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
        r2_val   = r2_score(y_val, y_val_pred)
        bias_val = np.mean(y_val - y_val_pred)
        med_ae   = median_absolute_error(y_val, y_val_pred)

        log(f"[TRAIN] R2={r2_tr:.4f} | RMSE={rmse_tr:.0f} | MAE={mae_tr:.0f} | Bias={bias_tr:.1f}")
        log(f"[VAL]   R2={r2_val:.4f} | RMSE={rmse_val:.0f} | MAE={mae_val:.0f} | Bias={bias_val:.1f}")

        fold_maes_tr.append(mae_tr)
        fold_rmses_tr.append(rmse_tr)
        fold_r2s_tr.append(r2_tr)
        fold_bias_tr.append(bias_tr)

        fold_maes_val.append(mae_val)
        fold_rmses_val.append(rmse_val)
        fold_r2s_val.append(r2_val)
        fold_bias_val.append(bias_val)
        fold_med_ae.append(med_ae)

    # =====================================================
    # FINAL RESULTS
    # =====================================================

    log("\n===== FINAL CROSS-VALIDATION RESULTS =====")
    log(f"[TRAIN AVG] MAE={np.mean(fold_maes_tr):.1f} | "
        f"R2={np.mean(fold_r2s_tr):.4f} | Bias={np.mean(fold_bias_tr):.1f}")
    log(f"[VAL AVG]   MAE={np.mean(fold_maes_val):.1f} | "
        f"R2={np.mean(fold_r2s_val):.4f} | "
        f"Bias={np.mean(fold_bias_val):.1f} | "
        f"RMSE={np.mean(fold_rmses_val):.1f}")


feature engineering com tudo, sem selection 

In [ ]:
# ---------------------------------------------------------
# 0) LOGGING
# ---------------------------------------------------------
LOG_FILE = "mean_fe_no_fs_.log"

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)

logging.info(f"SINGLE CONFIG | {N_SPLITS}-fold | HGB + RF + ET | FE | NO FS")

# ---------------------------------------------------------
# 1) CONFIGURAÇÃO ÚNICA
# ---------------------------------------------------------
CONFIG_NAME = "HGB_RF_ET_FE_NO_FS"

PARAMS = {
    "hgb": {
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

# ---------------------------------------------------------
# 2) K-FOLD
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(
                df,
                column=col,
                remove_middle_spaces=True,
                allow_extra_chars=""
            )
    return df

# ---------------------------------------------------------
# 3) EVAL DA CONFIGURAÇÃO
# ---------------------------------------------------------
def eval_single_config(name: str, params: dict) -> dict:
    fold_rows = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"[{name}] Fold {fold}/{N_SPLITS}")
        logging.info(f"[{name}] Fold {fold}/{N_SPLITS}")

        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        # Log target
        y_train_log = np.log1p(y_train)

        # Normalização de strings
        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        # Base columns
        base_cols = [c for c in (numeric_features + categorical_features) if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        required = ["year", "mileage", "engineSize", "tax", "mpg", "previousOwners", "Brand", "model"]
        missing = [c for c in required if c not in X_train.columns]
        if missing:
            raise KeyError(f"[Fold {fold}] faltam colunas necessárias no pipeline: {missing}")

        # -----------------------------------------------------
        # A) CLEANING
        # -----------------------------------------------------
        year_state = fit_year_median(X_train, "year", "model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val,   year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val,   mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val,   engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val,   mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   owners_state)

        # -----------------------------------------------------
        # B) RESOLVERS
        # -----------------------------------------------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # -----------------------------------------------------
        # C) FEATURE ENGINEERING
        # -----------------------------------------------------
        X_train = add_owners_flagged(X_train, "previousOwners", "owners_flagged", drop_original=True)
        X_val   = add_owners_flagged(X_val,   "previousOwners", "owners_flagged", drop_original=True)

        X_train = create_age_and_drop_year(X_train, "year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   "year", base_year=2020)

        X_train = add_mileage_features(X_train, "mileage", "age", drop_original=True, drop_ratio=True)
        X_val   = add_mileage_features(X_val,   "mileage", "age", drop_original=True, drop_ratio=True)

        X_train = add_engine_bins(X_train, "engineSize", "engine_bin")
        X_val   = add_engine_bins(X_val,   "engineSize", "engine_bin")

        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # -----------------------------------------------------
        # D) ENCODING
        # -----------------------------------------------------
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low], axis=1)

        drop_for_numeric = set(high_card_features + low_card_curr)
        numeric_curr = [c for c in X_train.columns if c not in drop_for_numeric]

        X_train_final = pd.concat([X_train[numeric_curr], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_curr],   X_val_cat], axis=1)

        scaler = StandardScaler()
        X_train_final[numeric_curr] = scaler.fit_transform(X_train_final[numeric_curr])
        X_val_final[numeric_curr]   = scaler.transform(X_val_final[numeric_curr])

        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # -----------------------------------------------------
        # E) MODELOS + ENSEMBLE
        # -----------------------------------------------------
        hgb = HistGradientBoostingRegressor(**params["hgb"])
        rf  = RandomForestRegressor(**params["rf"])
        et  = ExtraTreesRegressor(**params["et"])

        hgb.fit(X_train_final, y_train_log)
        rf.fit(X_train_final, y_train_log)
        et.fit(X_train_final, y_train_log)

        pred_tr = (
            0.6 * np.expm1(hgb.predict(X_train_final)) +
            0.4 * np.expm1(rf.predict(X_train_final))
        )

        pred_val = (
            0.6 * np.expm1(hgb.predict(X_val_final)) +
            0.4 * np.expm1(rf.predict(X_val_final))
        )

        # -----------------------------------------------------
        # F) MÉTRICAS
        # -----------------------------------------------------
        fold_rows.append({
            "fold": fold,
            "n_features": X_train_final.shape[1],
            "train_rmse": rmse(y_train, pred_tr),
            "val_rmse":   rmse(y_val,   pred_val),
            "train_mae":  mean_absolute_error(y_train, pred_tr),
            "val_mae":    mean_absolute_error(y_val,   pred_val),
            "train_r2":   r2_score(y_train, pred_tr),
            "val_r2":     r2_score(y_val,   pred_val),
            "train_bias": bias(y_train, pred_tr),
            "val_bias":   bias(y_val,   pred_val),
        })

    df = pd.DataFrame(fold_rows)

    return {
        "config": name,
        "val_rmse_mean": df["val_rmse"].mean(),
        "val_mae_mean":  df["val_mae"].mean(),
        "val_r2_mean":   df["val_r2"].mean(),
        "val_bias_mean": df["val_bias"].mean(),
        "train_rmse_mean": df["train_rmse"].mean(),
        "train_mae_mean":  df["train_mae"].mean(),
        "train_r2_mean":   df["train_r2"].mean(),
        "train_bias_mean": df["train_bias"].mean(),
        "avg_features": df["n_features"].mean(),
    }

# ---------------------------------------------------------
# 4) RUN
# ---------------------------------------------------------
print("\n-- Running single configuration --")
result = eval_single_config(CONFIG_NAME, PARAMS)

results_df = pd.DataFrame([result])
display(results_df)

OUT_CSV = "mean_fe_no_fs.csv"
results_df.to_csv(OUT_CSV, index=False)

print("\nFINAL RESULTS")
print(f"VAL RMSE: {result['val_rmse_mean']:.2f}")
print(f"VAL R2:   {result['val_r2_mean']:.4f}")
print(f"VAL MAE:  {result['val_mae_mean']:.2f}")
print(f"VAL Bias: {result['val_bias_mean']:.2f}")
print(f"Avg Features: {result['avg_features']:.0f}")

print(f"\nResults saved to: {OUT_CSV}")
print(f"Log file: {LOG_FILE}")


fe com fs, 65 percentagem

In [ ]:
# =========================================================
# SINGLE CONFIG — FE + FS + BAGGING
# =========================================================

FS_KEEP_RATIO = 0.65
CONFIG_NAME = "FE_FS65_SINGLE_CONFIG"

# ---------------------------------------------------------
# 0) FEATURE SELECTION MODEL (RF)
# ---------------------------------------------------------
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# ---------------------------------------------------------
# 1) MODELS PARAMS — FIXOS
# ---------------------------------------------------------
PARAMS = {
    "hgb": {
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

# ---------------------------------------------------------
# 2) LOGGING
# ---------------------------------------------------------
LOG_FILE = "mean_fe_fs_65.log"
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(message)s",
    filemode="w",
    force=True
)
logging.info(f"{CONFIG_NAME} | FS_KEEP_RATIO={FS_KEEP_RATIO} | {N_SPLITS}-fold")

# ---------------------------------------------------------
# 3) K-FOLD
# ---------------------------------------------------------
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

def _normalize_cats(df):
    df = df.copy()
    for col in cols_to_normalize:
        if col in df.columns:
            df[col] = fill_unknown(df[col])
            df = column_string_transformer(df, col, True, "")
    return df

# ---------------------------------------------------------
# 4) EVAL SINGLE CONFIG
# ---------------------------------------------------------
def eval_single_config(name: str, params: dict) -> dict:
    rows = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X), start=1):
        print(f"[{name}] Fold {fold}/{N_SPLITS}")

        X_train = X.iloc[tr_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[tr_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        y_train_log = np.log1p(y_train)

        X_train = _normalize_cats(X_train)
        X_val   = _normalize_cats(X_val)

        base_cols = [c for c in numeric_features + categorical_features if c in X_train.columns]
        X_train = X_train[base_cols].copy()
        X_val   = X_val[base_cols].copy()

        # ---------------- CLEANING ----------------
        year_state = fit_year_median(X_train, "year", "model")
        X_train = transform_year_with_model_median(X_train, year_state)
        X_val   = transform_year_with_model_median(X_val,   year_state)

        mileage_state = fit_mileage_imputer(X_train, "mileage", True)
        X_train = transform_mileage_imputer(X_train, mileage_state)
        X_val   = transform_mileage_imputer(X_val,   mileage_state)

        engine_state = fit_engine_size_imputer(X_train, "engineSize")
        X_train = transform_engine_size_imputer(X_train, engine_state)
        X_val   = transform_engine_size_imputer(X_val,   engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, "mpg", True)
        X_train = transform_mpg_imputer(X_train, mpg_state)
        X_val   = transform_mpg_imputer(X_val,   mpg_state)

        owners_state = fit_previous_owners_imputer(X_train, "previousOwners", "year", "mileage")
        X_train = transform_previous_owners_imputer(X_train, owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   owners_state)

        # ---------------- RESOLVERS ----------------
        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val,   _, _ = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

        # ---------------- FE ----------------
        X_train = add_owners_flagged(X_train, "previousOwners", "owners_flagged", True)
        X_val   = add_owners_flagged(X_val,   "previousOwners", "owners_flagged", True)

        X_train = create_age_and_drop_year(X_train, "year", 2020)
        X_val   = create_age_and_drop_year(X_val,   "year", 2020)

        X_train = add_mileage_features(X_train, "mileage", "age", True, True)
        X_val   = add_mileage_features(X_val,   "mileage", "age", True, True)

        X_train = add_engine_bins(X_train, "engineSize", "engine_bin")
        X_val   = add_engine_bins(X_val,   "engineSize", "engine_bin")

        X_train["engine_bin"] = X_train["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
        X_val["engine_bin"]   = X_val["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

        low_card_curr = low_card_features + ["engine_bin"]

        # ---------------- ENCODING ----------------
        te = MyTargetEncoder(5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_curr])
        X_train_low = ohe.transform(X_train[low_card_curr])
        X_val_low   = ohe.transform(X_val[low_card_curr])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low], axis=1)

        drop_cols = set(high_card_features + low_card_curr)
        num_cols = [c for c in X_train.columns if c not in drop_cols]

        X_train_final = pd.concat([X_train[num_cols], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[num_cols],   X_val_cat], axis=1)

        scaler = StandardScaler()
        X_train_final[num_cols] = scaler.fit_transform(X_train_final[num_cols])
        X_val_final[num_cols]   = scaler.transform(X_val_final[num_cols])

        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        # ---------------- FEATURE SELECTION ----------------
        n_feats = X_train_final.shape[1]
        k = max(1, int(np.ceil(FS_KEEP_RATIO * n_feats)))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            rf_fs,
            max_features=k,
            threshold=-np.inf,
            prefit=True
        )

        cols_sel = X_train_final.columns[selector.get_support()]
        X_train_sel = X_train_final[cols_sel]
        X_val_sel   = X_val_final[cols_sel]

        # ---------------- MODELS ----------------
        hgb = HistGradientBoostingRegressor(**params["hgb"])
        rf  = RandomForestRegressor(**params["rf"])
        et  = ExtraTreesRegressor(**params["et"])

        hgb.fit(X_train_sel, y_train_log)
        rf.fit(X_train_sel, y_train_log)
        et.fit(X_train_sel, y_train_log)

        pred_tr = (
            0.6 * np.expm1(hgb.predict(X_train_sel)) +
            0.4 * np.expm1(rf.predict(X_train_sel))
        )
        pred_val = (
            0.6 * np.expm1(hgb.predict(X_val_sel)) +
            0.4 * np.expm1(rf.predict(X_val_sel))
        )

        rows.append({
            "fold": fold,
            "n_total": n_feats,
            "n_selected": len(cols_sel),
            "rmse_val": rmse(y_val, pred_val),
            "mae_val": mean_absolute_error(y_val, pred_val),
            "r2_val": r2_score(y_val, pred_val),
            "bias_val": bias(y_val, pred_val)
        })

    df = pd.DataFrame(rows)

    return {
        "config": name,
        "val_rmse": df.rmse_val.mean(),
        "val_mae": df.mae_val.mean(),
        "val_r2": df.r2_val.mean(),
        "val_bias": df.bias_val.mean(),
        "avg_selected_features": df.n_selected.mean(),
        "avg_total_features": df.n_total.mean()
    }

# ---------------------------------------------------------
# 5) RUN
# ---------------------------------------------------------
result = eval_single_config(CONFIG_NAME, PARAMS)
display(pd.DataFrame([result]))

print("\nFINAL RESULTS")
for k, v in result.items():
    if k != "config":
        print(f"{k}: {v:.4f}")

print(f"\nLog saved to: {LOG_FILE}")


sem previous owners, log do price,age - sem feature selection

In [ ]:
# =========================================
# SINGLE CONFIG PIPELINE (LOG TARGET + AGE)
# =========================================
#este numeric features está certo?
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]
DROP_FROM_MODEL = ["previousOwners"]

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# FIXED PARAMETERS (uma configuração fixa)
params = {
    "hgb": {
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

search_results = []
best_rmse = np.inf
best_config = None

# log setup
log_dir = "mean_logs"
os.makedirs(log_dir, exist_ok=True)
log_filename = "no_fs_age_log.txt"
log_path = os.path.join(log_dir, log_filename)

with open(log_path, "w", encoding="utf-8") as log_file:
    
    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()
    
    log("# =============================")
    log("# START OF SINGLE CONFIG PIPELINE (LOG TARGET + AGE FEATURE)")
    log("# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")
    
    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

        y_train_log = np.log1p(y_train)
        log(f"\n[FOLD {fold}] Processing fold...")

        # ---------------- Preprocessing / FE ----------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, state=mileage_state)
        X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val,   fuel_state)

        # FE: age
        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        # drop excluded features
        X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
        X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

        # ---------------- Encoding ----------------
        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high, X_val_low], axis=1)

        # join numeric + categorical
        X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

        if len(numeric_features) > 0:
            scaler = StandardScaler()
            X_train_final[numeric_features] = scaler.fit_transform(X_train_final[numeric_features])
            X_val_final[numeric_features]   = scaler.transform(X_val_final[numeric_features])

        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        if fold == 1:
            feature_names = list(X_train_final.columns)
            log(f"  > Features Used ({len(feature_names)}): {feature_names}")

        # ---------------- Train Models ----------------
        hgb_model = HistGradientBoostingRegressor(**params["hgb"])
        rf_model  = RandomForestRegressor(**params["rf"])
        et_model  = ExtraTreesRegressor(**params["et"])
        hgb_model.fit(X_train_final, y_train_log)
        rf_model.fit(X_train_final, y_train_log)
        et_model.fit(X_train_final, y_train_log)

        # Predictions
        y_tr_hgb  = np.expm1(hgb_model.predict(X_train_final))
        y_val_hgb = np.expm1(hgb_model.predict(X_val_final))

        y_tr_rf  = np.expm1(rf_model.predict(X_train_final))
        y_val_rf = np.expm1(rf_model.predict(X_val_final))

        y_tr_et  = np.expm1(et_model.predict(X_train_final))
        y_val_et = np.expm1(et_model.predict(X_val_final))

        pred_tr = 0.6*y_tr_hgb + 0.4*y_tr_rf
        pred_val = 0.6*y_val_hgb + 0.4*y_val_rf

        # Metrics
        mae_tr  = mean_absolute_error(y_train, pred_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, pred_tr))
        r2_tr   = r2_score(y_train, pred_tr)
        bias_tr = np.mean(y_train - pred_tr)

        mae_val  = mean_absolute_error(y_val, pred_val)
        rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
        r2_val   = r2_score(y_val, pred_val)
        bias_val = np.mean(y_val - pred_val)

        fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
        fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
        fold_med_ae.append(median_absolute_error(y_val, pred_val))

        log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
        log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")

    # ---------------- Summary ----------------
    mean_rmse_val = np.mean(fold_rmses_val)
    mean_mae_val  = np.mean(fold_maes_val)
    mean_r2_val   = np.mean(fold_r2s_val)
    mean_bias_val = np.mean(fold_bias_val)

    mean_mae_tr   = np.mean(fold_maes_tr)
    mean_r2_tr    = np.mean(fold_r2s_tr)
    mean_bias_tr  = np.mean(fold_bias_tr)

    log("\nSUMMARY ACROSS FOLDS:")
    log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
    log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

    search_results.append({
        "config": "single_config_age",
        "val_rmse": mean_rmse_val,
        "val_mae": mean_mae_val,
        "val_r2": mean_r2_val,
        "val_bias": mean_bias_val,
        "train_mae": mean_mae_tr,
        "train_r2": mean_r2_tr,
        "train_bias": mean_bias_tr,
        "val_med_ae": np.mean(fold_med_ae)
    })

log("# END OF PIPELINE")

results_df = pd.DataFrame(search_results)
display(results_df)


sem previous owners, log do price,age - com fs -80 %

In [ ]:
# =========================================
# SINGLE CONFIG PIPELINE (LOG TARGET + AGE + FS 90%)
# =========================================
#estas numeric features estão bem?
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]
DROP_FROM_MODEL = ["previousOwners"]
FS_KEEP_RATIO = 0.80

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# RandomForest used only to rank feature importances for SelectFromModel
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

# FIXED MODEL PARAMETERS
params = {
    "hgb": {
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    },
    "rf": {
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    },
    "et": {
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }
}

search_results = []
best_rmse = np.inf
best_config = None

# log setup
log_dir = "mean_logs"
os.makedirs(log_dir, exist_ok=True)
log_filename = "mean_age_fs80_log.txt"
log_path = os.path.join(log_dir, log_filename)

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# START OF SINGLE CONFIG PIPELINE (LOG TARGET + AGE + FS 80%)")
    log("# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")

    fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
    fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
    fold_med_ae = []
    fold_nsel = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

        y_train_log = np.log1p(y_train)
        log(f"\n[FOLD {fold}] Processing fold...")

        # ---------------- Preprocessing / FE ----------------
        year_state = fit_year_median(X_train, year_col="year", model_col="model")
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
        X_train = transform_mileage_imputer(X_train, state=mileage_state)
        X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

        engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
        X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

        mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
        X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
        X_val, _, _   = transform_ambiguous_brands(X_val,   brand_state)

        model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
        X_train, _, _ = transform_invalid_models(X_train, model_state)
        X_val, _, _   = transform_invalid_models(X_val,   model_state)

        transm_state = fit_transmission_resolver(X_train, valid_transmissions)
        X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
        X_val, _, _   = transform_transmission_resolver(X_val,   transm_state)

        fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
        X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
        X_val, _, _   = transform_fueltype_resolver(X_val,   fuel_state)

        # FE: age
        X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
        X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

        X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
        X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

        # ---------------- Encoding ----------------
        high_card_features = ["Brand", "model"]
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train_log)
        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])
        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high, X_val_low], axis=1)

        X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
        X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

        if len(numeric_features) > 0:
            scaler = StandardScaler()
            X_train_final[numeric_features] = scaler.fit_transform(X_train_final[numeric_features])
            X_val_final[numeric_features]   = scaler.transform(X_val_final[numeric_features])

        X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

        if fold == 1:
            feature_names = list(X_train_final.columns)
            log(f"  > Features before FS ({len(feature_names)}): {feature_names}")

        # ---------------- Feature Selection ----------------
        n_feats = X_train_final.shape[1]
        k = int(np.ceil(FS_KEEP_RATIO * n_feats))
        k = max(1, min(k, n_feats))

        rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
        rf_fs.fit(X_train_final, y_train_log)

        selector = SelectFromModel(
            estimator=rf_fs,
            threshold=-np.inf,
            max_features=k,
            prefit=True
        )

        selected_cols = X_train_final.columns[selector.get_support()]
        fold_nsel.append(len(selected_cols))

        X_train_sel = X_train_final[selected_cols]
        X_val_sel   = X_val_final[selected_cols]

        if fold == 1:
            log(f"  > Selected features ({len(selected_cols)}/{n_feats}) with FS_KEEP_RATIO={FS_KEEP_RATIO:.2f}")

        # ---------------- Train Models ----------------
        hgb_model = HistGradientBoostingRegressor(**params["hgb"])
        rf_model  = RandomForestRegressor(**params["rf"])
        et_model  = ExtraTreesRegressor(**params["et"])
        hgb_model.fit(X_train_sel, y_train_log)
        rf_model.fit(X_train_sel, y_train_log)
        et_model.fit(X_train_sel, y_train_log)

        # Predictions
        y_tr_hgb  = np.expm1(hgb_model.predict(X_train_sel))
        y_val_hgb = np.expm1(hgb_model.predict(X_val_sel))

        y_tr_rf  = np.expm1(rf_model.predict(X_train_sel))
        y_val_rf = np.expm1(rf_model.predict(X_val_sel))

        y_tr_et  = np.expm1(et_model.predict(X_train_sel))
        y_val_et = np.expm1(et_model.predict(X_val_sel))

        pred_tr = 0.6*y_tr_hgb + 0.4*y_tr_rf
        pred_val = 0.6*y_val_hgb + 0.4*y_val_rf

        # Metrics
        mae_tr  = mean_absolute_error(y_train, pred_tr)
        rmse_tr = np.sqrt(mean_squared_error(y_train, pred_tr))
        r2_tr   = r2_score(y_train, pred_tr)
        bias_tr = np.mean(y_train - pred_tr)

        mae_val  = mean_absolute_error(y_val, pred_val)
        rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
        r2_val   = r2_score(y_val, pred_val)
        bias_val = np.mean(y_val - pred_val)

        fold_maes_tr.append(mae_tr); fold_rmses_tr.append(rmse_tr); fold_r2s_tr.append(r2_tr); fold_bias_tr.append(bias_tr)
        fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
        fold_med_ae.append(median_absolute_error(y_val, pred_val))

        log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
        log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")

    # ---------------- Summary ----------------
    mean_rmse_val = np.mean(fold_rmses_val)
    mean_mae_val  = np.mean(fold_maes_val)
    mean_r2_val   = np.mean(fold_r2s_val)
    mean_bias_val = np.mean(fold_bias_val)

    mean_mae_tr   = np.mean(fold_maes_tr)
    mean_r2_tr    = np.mean(fold_r2s_tr)
    mean_bias_tr  = np.mean(fold_bias_tr)
    mean_nsel     = float(np.mean(fold_nsel))

    log("\nSUMMARY ACROSS FOLDS:")
    log(f"  Selected features (avg): {mean_nsel:.0f} / {X_train_final.shape[1]}")
    log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
    log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

    search_results.append({
        "config": "single_config_age_fs90",
        "fs_keep_ratio": FS_KEEP_RATIO,
        "avg_selected_features": mean_nsel,
        "val_rmse": mean_rmse_val,
        "val_mae": mean_mae_val,
        "val_r2": mean_r2_val,
        "val_bias": mean_bias_val,
        "train_mae": mean_mae_tr,
        "train_r2": mean_r2_tr,
        "train_bias": mean_bias_tr,
        "val_med_ae": np.mean(fold_med_ae)
    })

    log("# END OF PIPELINE")

results_df = pd.DataFrame(search_results)
display(results_df)


## 4. Randomized Hyperparameter Search with K-Fold Cross-Validation

In [11]:
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg'] # we removed previousOwners feature from here

In [ ]:
#MEAN COM LOG E AGE E NO PREV - 0.6/0.4, configs random forest diferentes 
#melhor- manter

# numeric features used by the model (year is replaced by age; previousOwners is excluded)
numeric_features = ["mileage", "engineSize", "tax", "mpg", "age"]

# we will use an 8-fold cross-validation strategy
N_SPLITS = 8
# shuffles the data, in order to (at least theoretically) have more balanced folds
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# we explicitly exclude this feature from the model input
DROP_FROM_MODEL = ["previousOwners"]

# hyperparameter space that we are going to use for random search
# each configuration will get 1 value from each of these lists
param_distributions = {
    "hgb": [{
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    }],
    "rf": [{
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }],
    "et": [{
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }],
}
FS_KEEP_RATIO = 1.0  # keep top 90% features by RF importance
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

N_RANDOM_CONFIGS = 1    # short random search, enough for baseline results

# we get N_RANDOM_CONFIGS random parameter combinations from param_distributions
sampler = ParameterSampler(param_distributions, n_iter=N_RANDOM_CONFIGS, random_state=RANDOM_STATE)

search_results = []  # stores results for ranking
best_rmse = np.inf
best_config = None

# log setup (mainly for development and traceability)
log_dir = "MEAN_logs"
os.makedirs(log_dir, exist_ok=True)
log_filename = "MEAN_logprice_log2.txt"
log_path = os.path.join(log_dir, log_filename)

# Note: this section is included mainly for organizational purposes
with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# START OF HGB SEARCH (LOG TARGET + AGE FEATURE)")
    log("# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS")
    log(f"# LOG FILE SAVED TO: {log_path}")
    log("# =============================")

    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        fold_rmses_val, fold_maes_val, fold_r2s_val, fold_bias_val = [], [], [], []
        fold_rmses_tr,  fold_maes_tr,  fold_r2s_tr,  fold_bias_tr  = [], [], [], []
        fold_med_ae = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
            y_train, y_val = y.iloc[train_idx].copy(), y.iloc[val_idx].copy()

            # log-transform target for training
            y_train_log = np.log1p(y_train)

            log(f"\n[C{config_id}|F{fold}] Processing fold...")

            # -> Preprocessing steps (fit only on training fold, apply to both)
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            X_train = transform_tax_custom_rules(X_train, "tax", "year", "fuelType", "engineSize")
            X_val   = transform_tax_custom_rules(X_val,   "tax", "year", "fuelType", "engineSize")

            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            brand_state = fit_ambiguous_brand_resolver(X_train, valid_brands)
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val,   _, _ = transform_ambiguous_brands(X_val,   brand_state)

            model_state = fit_invalid_model_resolver(X_train, valid_models_by_brand)
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val,   _, _ = transform_invalid_models(X_val,   model_state)

            transm_state = fit_transmission_resolver(X_train, valid_transmissions)
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val,   _, _ = transform_transmission_resolver(X_val,   transm_state)

            fuel_state = fit_fueltype_resolver(X_train, valid_fueltypes)
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val,   _, _ = transform_fueltype_resolver(X_val,   fuel_state)

            # -> Feature engineering: create age and drop year (do this after all year-based steps)
            X_train = create_age_and_drop_year(X_train, year_col="year", base_year=2020)
            X_val   = create_age_and_drop_year(X_val,   year_col="year", base_year=2020)

            # hard guarantee: previousOwners is not used by the model
            X_train = X_train.drop(columns=DROP_FROM_MODEL, errors="ignore")
            X_val   = X_val.drop(columns=DROP_FROM_MODEL, errors="ignore")

            # -> Feature encoding (categorical only)
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train_log)  # fit using log-target
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # join numerical and categorical (encoded) features
            X_train_final = pd.concat([X_train[numeric_features], X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val[numeric_features],   X_val_cat],   axis=1)

            # align columns to avoid mismatch due to encoding
            X_val_final = X_val_final.reindex(columns=X_train_final.columns, fill_value=0)

            if fold == 1:
                feature_names = list(X_train_final.columns)
                log(f"  > Features Used ({len(feature_names)}): {feature_names}")

            # Scaling
            # -------------------------
            scaler = StandardScaler()
            X_train_final = pd.DataFrame(scaler.fit_transform(X_train_final), index=X_train_final.index, columns=X_train_final.columns)
            X_val_final   = pd.DataFrame(scaler.transform(X_val_final),   index=X_val_final.index,   columns=X_val_final.columns)

            # -------------------------
            # Feature Selection
            # -------------------------
            n_feats = X_train_final.shape[1]
            k = int(np.ceil(FS_KEEP_RATIO * n_feats))
            k = max(1, min(k, n_feats))

            rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
            rf_fs.fit(X_train_final, y_train_log)

            selector = SelectFromModel(
                estimator=rf_fs,
                threshold=-np.inf,
                max_features=k,
                prefit=True
            )

            selected_cols = X_train_final.columns[selector.get_support()]
            X_train_sel = X_train_final[selected_cols]
            X_val_sel   = X_val_final[selected_cols]

            # -------------------------
            # Train Mean Ensemble (HGB log + RF/ET normal)
            # -------------------------
            hgb_model = HistGradientBoostingRegressor(**params["hgb"])
            hgb_model.fit(X_train_sel, y_train_log)

            # RF and ET on original target
            rf_model  = RandomForestRegressor(**params["rf"])
            et_model  = ExtraTreesRegressor(**params["et"])
            rf_model.fit(X_train_sel, y_train_log)
            et_model.fit(X_train_sel, y_train_log)

            # Predictions
            y_tr_hgb_log  = hgb_model.predict(X_train_sel)
            y_val_hgb_log = hgb_model.predict(X_val_sel)

            y_tr_rf_log   = rf_model.predict(X_train_sel)
            y_val_rf_log  = rf_model.predict(X_val_sel)

            y_tr_et_log   = et_model.predict(X_train_sel)
            y_val_et_log  = et_model.predict(X_val_sel)


            # Invert log-transform
            y_tr_hgb  = np.expm1(y_tr_hgb_log)
            y_val_hgb = np.expm1(y_val_hgb_log)

            y_tr_rf   = np.expm1(y_tr_rf_log)
            y_val_rf  = np.expm1(y_val_rf_log)

            y_tr_et   = np.expm1(y_tr_et_log)
            y_val_et  = np.expm1(y_val_et_log)


            # Mean ensemble - melhor do kaggle
            y_pred_train = 0.6*y_tr_hgb + 0.4*y_tr_rf + 0*y_tr_et
            y_pred_val   = 0.6*y_val_hgb + 0.4*y_val_rf + 0*y_val_et


            # -> Metrics on original scale
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_train))
            r2_tr   = r2_score(y_train, y_pred_train)
            bias_tr = np.mean(y_train - y_pred_train)

            mae_val  = mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
            r2_val   = r2_score(y_val, y_pred_val)
            bias_val = np.mean(y_val - y_pred_val)

            med_ae_val = median_absolute_error(y_val, y_pred_val)

            log(f"  > [TRAIN] R2: {r2_tr:.4f} | RMSE: {rmse_tr:.0f} | MAE: {mae_tr:.0f} | Bias: {bias_tr:.1f}")
            log(f"  > [VAL]   R2: {r2_val:.4f} | RMSE: {rmse_val:.0f} | MAE: {mae_val:.0f} | Bias: {bias_val:.1f}")

            fold_maes_tr.append(mae_tr);   fold_rmses_tr.append(rmse_tr);   fold_r2s_tr.append(r2_tr);   fold_bias_tr.append(bias_tr)
            fold_maes_val.append(mae_val); fold_rmses_val.append(rmse_val); fold_r2s_val.append(r2_val); fold_bias_val.append(bias_val)
            fold_med_ae.append(med_ae_val)

        mean_rmse_val = np.mean(fold_rmses_val)
        mean_mae_val  = np.mean(fold_maes_val)
        mean_r2_val   = np.mean(fold_r2s_val)
        mean_bias_val = np.mean(fold_bias_val)

        mean_mae_tr   = np.mean(fold_maes_tr)
        mean_r2_tr    = np.mean(fold_r2s_tr)
        mean_bias_tr  = np.mean(fold_bias_tr)

        log("")
        log(f"Config {config_id} SUMMARY:")
        log(f"  [TRAIN AVG] MAE: {mean_mae_tr:.1f} | R2: {mean_r2_tr:.4f} | Bias: {mean_bias_tr:.1f}")
        log(f"  [VAL AVG]   MAE: {mean_mae_val:.1f} | R2: {mean_r2_val:.4f} | Bias: {mean_bias_val:.1f} | RMSE: {mean_rmse_val:.1f}")

        search_results.append({
            "config_id": config_id,
            **params,

            "val_rmse": mean_rmse_val,
            "val_mae": mean_mae_val,
            "val_r2": mean_r2_val,
            "val_bias": mean_bias_val,

            "train_mae": mean_mae_tr,
            "train_r2": mean_r2_tr,
            "train_bias": mean_bias_tr,

            "val_med_ae": np.mean(fold_med_ae)
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

    log("# END OF SEARCH")

# Results
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="val_rmse", ascending=True)

print("\nTop 5 Configurations by RMSE:")
display(results_df_sorted.head(5))

print("\nBest Config found:")
print(best_config)


# =============================
# START OF HGB SEARCH (LOG TARGET + AGE FEATURE)
# METRICS ON ORIGINAL SCALE: R2, RMSE, MAE, BIAS
# LOG FILE SAVED TO: MEAN_logs\MEAN_logprice_log2.txt
# =============================

######## CONFIG 1/1 ########
Parameters: {'rf': {'n_estimators': 200, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True, 'random_state': 42, 'n_jobs': -1}, 'hgb': {'max_iter': 1200, 'learning_rate': 0.07, 'max_depth': 20, 'max_leaf_nodes': 191, 'min_samples_leaf': 16, 'l2_regularization': 3.0, 'loss': 'squared_error', 'random_state': 42}, 'et': {'n_estimators': 1200, 'max_depth': 30, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.5, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1}}

[C1|F1] Processing fold...
  > Features Used (15): ['mileage', 'engineSize', 'tax', 'mpg', 'age', 'Brand', 'model', 'transmission_AUTOMATIC', 'transmission_MANUAL', 'transmission_SEMIAUTO', 'fuelType_DIESEL', 'fuelType_ELECTRIC', 'fuelType_HYB

,config_id,rf,hgb,et,val_rmse,val_mae,val_r2,val_bias,train_mae,train_r2,train_bias,val_med_ae
0,1,"{'n_estimators': 200, 'min_samples_leaf': 1, '...","{'max_iter': 1200, 'learning_rate': 0.07, 'max...","{'n_estimators': 1200, 'max_depth': 30, 'min_s...",2106.395903,1219.795078,0.953004,113.82188,830.060005,0.97999,79.962926,762.55737



Best Config found:
{'rf': {'n_estimators': 200, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True, 'random_state': 42, 'n_jobs': -1}, 'hgb': {'max_iter': 1200, 'learning_rate': 0.07, 'max_depth': 20, 'max_leaf_nodes': 191, 'min_samples_leaf': 16, 'l2_regularization': 3.0, 'loss': 'squared_error', 'random_state': 42}, 'et': {'n_estimators': 1200, 'max_depth': 30, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.5, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1}}


final config e submissao no kaggle

In [32]:

numeric_features = ["year", "mileage", "engineSize", "tax", "mpg"]

RANDOM_STATE = 42

final_params = {
    "hgb": [{
        "max_iter": 1200,
        "learning_rate": 0.07,
        "max_depth": 20,
        "max_leaf_nodes": 191,
        "min_samples_leaf": 16,
        "l2_regularization": 3.0,
        "loss": "squared_error",
        "random_state": RANDOM_STATE
    }],
    "rf": [{
        "n_estimators": 200,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "max_depth": 20,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }],
    "et": [{
        "n_estimators": 1200,
        "max_depth": 30,
        "min_samples_split": 4,
        "min_samples_leaf": 1,
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }]
}

# FS_KEEP_RATIO=1 => keeps all features (no real selection), but kept here to match your structure
FS_KEEP_RATIO = 1
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

print(f"- using params: {final_params}")
print(f"- FS_KEEP_RATIO: {FS_KEEP_RATIO}")

# ==============================================================================
# 1) LOAD TEST + IDS
# ==============================================================================
try:
    test_df_raw = pd.read_csv("test.csv")
except:
    test_df_raw = pd.read_csv("../../project_data/test.csv")

ID_CANDIDATES = ["id", "carID", "carId", "car_id"]
ID_COL = next((c for c in ID_CANDIDATES if c in test_df_raw.columns), None)

if ID_COL is not None:
    test_ids = test_df_raw[ID_COL].copy()
    test_df = test_df_raw.drop(columns=[ID_COL]).copy()
else:
    test_ids = test_df_raw.index.copy()
    test_df = test_df_raw.copy()

# ==============================================================================
# 2) START DATA (LOG TARGET)
# ==============================================================================
X_full = X.copy()
y_full = y.copy()
y_full_log = np.log1p(y_full)  # LOG TARGET (as requested)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]

# IMPORTANT: you are NOT using engine_bin anymore, so low_card_curr is just low_card_features
low_card_curr = low_card_features

# ==============================================================================
# 3) STRING NORMALIZATION (TRAIN + TEST)
# ==============================================================================
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer(
            X_full, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )
    if col in test_df.columns:
        test_df[col] = fill_unknown(test_df[col])
        test_df = column_string_transformer(
            test_df, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )

# Restrict to expected columns (train schema)
expected_cols = [c for c in (numeric_features + categorical_features) if c in X_full.columns]
X_full = X_full[expected_cols].copy()
X_test = test_df.reindex(columns=expected_cols, fill_value=np.nan).copy()

# ==============================================================================
# 4) FIT + TRANSFORM ON FULL TRAIN
# ==============================================================================
print("- fitting transforms on full train")

year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

X_full = transform_tax_custom_rules(X_full, "tax", "year", "fuelType", "engineSize")

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# ==============================================================================
# 5) FEATURE ENGINEERING: ONLY AGE (drop year)
# ==============================================================================
print("- creating engineered features on full train")
X_full = create_age_and_drop_year(X_full, year_col="year", base_year=2020)

# ==============================================================================
# 6) ENCODING (FIT ON FULL TRAIN) using LOG TARGET
# ==============================================================================
print("- fitting encoders on full train")

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full_log)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_curr])
X_full_low = ohe.transform(X_full[low_card_curr])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr = [c for c in X_full.columns if c not in drop_for_numeric]

X_full_final = pd.concat([X_full[numeric_features_curr], X_full_cat], axis=1)
print(f"- train matrix shape: {X_full_final.shape}")

# ==============================================================================
# 7) FEATURE SELECTION (kept but FS_KEEP_RATIO=1 => keeps all)
# ==============================================================================
print("- fitting feature selector (RF)")

n_feats = X_full_final.shape[1]
k = int(np.ceil(FS_KEEP_RATIO * n_feats))
k = max(1, min(k, n_feats))

rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
rf_fs.fit(X_full_final, y_full_log)

selector = SelectFromModel(
    estimator=rf_fs,
    threshold=-np.inf,
    max_features=k,
    prefit=True
)

selected_cols = X_full_final.columns[selector.get_support()]
X_full_sel = X_full_final[selected_cols]
print(f"- selected features: {len(selected_cols)}/{n_feats}")

# ==============================================================================
# 8) TRAIN FINAL MODELS (LOG TARGET) — MEAN ENSEMBLE
# ==============================================================================
print("- training final models (HGB + RF + ET)")

hgb_final = HistGradientBoostingRegressor(
    **final_params["hgb"][0]
)
hgb_final.fit(X_full_sel, y_full_log)

rf_final = RandomForestRegressor(
    **final_params["rf"][0]
)
rf_final.fit(X_full_sel, y_full_log)

et_final = ExtraTreesRegressor(
    **final_params["et"][0]
)
et_final.fit(X_full_sel, y_full_log)

print("- models trained")

# ==============================================================================
# 9) TRANSFORM TEST (TRANSFORM-ONLY) + SAME AGE + SAME ENCODING + SAME FS
# ==============================================================================
print("- transforming test set")

X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_custom_rules(X_test, "tax", "year", "fuelType", "engineSize")
X_test = transform_mpg_imputer(X_test, state=mpg_state)

X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# same FE: ONLY AGE
X_test = create_age_and_drop_year(X_test, year_col="year", base_year=2020)

# encoding (transform-only)
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_curr])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr_test = [c for c in X_test.columns if c not in drop_for_numeric]

X_test_final = pd.concat(
    [X_test[numeric_features_curr_test], X_test_cat],
    axis=1
)

# align to train columns
X_test_final = X_test_final.reindex(
    columns=X_full_final.columns,
    fill_value=0
)

# apply SAME feature selection
X_test_sel = X_test_final.reindex(
    columns=selected_cols,
    fill_value=0
)

print(f"- test matrix shape: {X_test_sel.shape}")

# ==============================================================================
# 10) PREDICT + MEAN ENSEMBLE + INVERT LOG
# ==============================================================================
print("- predicting (mean ensemble)")

pred_hgb_log = hgb_final.predict(X_test_sel)
pred_rf_log  = rf_final.predict(X_test_sel)
pred_et_log  = et_final.predict(X_test_sel)

# mean in LOG space
pred_log_mean = (
    0.5 * pred_hgb_log +
    0.3 * pred_rf_log +
    0.2 * pred_et_log
)

pred_final = np.expm1(pred_log_mean)
pred_final = np.maximum(pred_final, 0)

submission = pd.DataFrame({
    (ID_COL if ID_COL is not None else "id"): test_ids,
    "price": pred_final
})

submission_filename = "submission_0.5_0.3_mean_ensemble_log_age.csv"
submission.to_csv(submission_filename, index=False)

print(f"- saved: {submission_filename}")
print(submission.head())


- using params: {'hgb': [{'max_iter': 1200, 'learning_rate': 0.07, 'max_depth': 20, 'max_leaf_nodes': 191, 'min_samples_leaf': 16, 'l2_regularization': 3.0, 'loss': 'squared_error', 'random_state': 42}], 'rf': [{'n_estimators': 200, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True, 'random_state': 42, 'n_jobs': -1}], 'et': [{'n_estimators': 1200, 'max_depth': 30, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 0.5, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1}]}
- FS_KEEP_RATIO: 1
- fitting transforms on full train
- creating engineered features on full train
- fitting encoders on full train
- train matrix shape: (75973, 15)
- fitting feature selector (RF)
- selected features: 15/15
- training final models (HGB + RF + ET)
- models trained
- transforming test set
- test matrix shape: (32567, 15)
- predicting (mean ensemble)
- saved: submission_0.5_0.3_mean_ensemble_log_age.csv
    carID         price
0   89856   8390.084541
1  106581 